# 🏛️ MARINERA POWER: Pipeline Industrial Definitivo & Control Humano (Lotes de 20)

Pipeline oficial de profesionalización de catálogo para **Marinera Power**.
- **Fondo de Estudio:** Gris cálido `#E5E1DE` con sombra de contacto y luz difusa 5200K.
- **Pre-procesamiento:** Corrección EXIF + Detección de bordes Sobel (umbral 3.5) + Redimensionamiento Lanczos a 1920px.
- **Control Humano (Human-in-the-Loop):** Procesamiento en **lotes de 20 fotos** para revisión y feedback de Carlos Carbajal.
- **Respaldo Doble:** Enrutamiento en espejo a carpetas PRO / Erróneas + Bóveda Central permanente.
- **Tablero Web en Vivo:** https://carloscarbajallinan.github.io/marinera-studio-pro/


In [ ]:
# 1. Montar Google Drive & Dependencias Industriales
from google.colab import drive
drive.mount('/content/drive')

!pip install -q -U pillow pillow-heif requests google-auth gspread opencv-python-headless numpy
import pillow_heif
pillow_heif.register_heif_opener()
print('✅ Google Drive montado y dependencias industriales listas.')

In [ ]:
# 2. Autenticación, Conexión Sheets y Auto-Detección de Rutas en Drive
import os, io, gc, json, base64, time, requests, shutil
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageOps
from google.colab import userdata, auth
from IPython.display import display, Image as IPImage
import gspread
from google.auth import default

# 1. Cargar API Key
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    print('✅ GEMINI_API_KEY cargada con éxito desde Secrets.')
except Exception:
    import getpass
    API_KEY = getpass.getpass('Ingresa tu Gemini API Key: ')

# 2. Conexión Google Sheets
auth.authenticate_user()
creds, _ = default()
gc_client = gspread.authorize(creds)
SPREADSHEET_ID = '1kRsCZ7ZpPxyxrSEkUrYRl3NHahrQQche90RKK2yedQ8'
sh = gc_client.open_by_key(SPREADSHEET_ID)
sheet_control = sh.worksheet('Control_Total_Drive_V2')
print(f'✅ Conexión establecida con la hoja maestra: {sheet_control.title}')

# 3. Auto-detección de la ruta exacta en Google Drive
candidates_base = [
    Path('/content/drive/MyDrive/Emprendimiento/Marinera power 2026'),
    Path('/content/drive/MyDrive/Marinera power 2026'),
    Path('/content/drive/MyDrive'),
]

BASE_DRIVE = Path('/content/drive/MyDrive')
for c in candidates_base:
    if (c / 'Fotos catalogo').exists():
        BASE_DRIVE = c
        break

DIR_ORIGEN_ROOT = BASE_DRIVE / 'Fotos catalogo'
DIR_PRO_ROOT = BASE_DRIVE / 'Fotos catalogo PRO'
DIR_ERROR_ROOT = DIR_PRO_ROOT / '_Fotos_Erroneas_y_Desalineadas'
BOVEDA_BACKUP_DIR = DIR_PRO_ROOT / '_BOVEDA_HISTORIAL_BACKUP'
CHECKPOINT_FILE = DIR_PRO_ROOT / 'checkpoint_pipeline.json'

DIR_PRO_ROOT.mkdir(parents=True, exist_ok=True)
DIR_ERROR_ROOT.mkdir(parents=True, exist_ok=True)
BOVEDA_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f'📁 Ruta Origen detectada:  {DIR_ORIGEN_ROOT}')
print(f'📁 Ruta PRO detectada:     {DIR_PRO_ROOT}')
print(f'📁 Ruta Erróneas en Drive: {DIR_ERROR_ROOT}')
print(f'📁 Bóveda de Respaldo:     {BOVEDA_BACKUP_DIR}')

# 4. Resolvedor Robusto Linux (partes_limpias + búsqueda insensible)
def resolve_photo_path(ruta_origen, orig_name):
    """
    Localiza la foto física en Google Drive de forma ultra-robusta.
    Limpia espacios en rutas tipo Linux y realiza búsqueda insensible a extensiones.
    """
    partes = [p.strip() for p in str(ruta_origen).split('/') if p.strip()]
    if partes and partes[0].lower() in ['fotos catalogo', 'fotos catalogo marinera power']:
        partes = partes[1:]
        
    foto_dir = DIR_ORIGEN_ROOT.joinpath(*partes)
    
    clean_name = str(orig_name).replace('.jpg', '').replace('.jpeg', '').replace('.png', '').replace('.heic', '').replace('.HEIC', '').strip()
    
    # 1. Búsqueda directa en la subcarpeta del producto
    if foto_dir.exists():
        cands = list(foto_dir.glob(f"{clean_name}*"))
        if cands:
            return cands[0]
            
    # 2. Fallback global insensible en todo el árbol de origen
    global_cands = list(DIR_ORIGEN_ROOT.glob(f"**/{clean_name}*"))
    if global_cands:
        return global_cands[0]
        
    return None

print('✅ Resolvedor de rutas Linux y búsqueda insensible compilado.')



In [ ]:
# 3. Pre-Procesamiento Sobel (Umbral 3.5) & Context Managers Anti-OOM
def detect_and_crop_artificial_bars_sobel(img_rgb: Image.Image, max_scan_pct: float = 0.25):
    arr = np.array(img_rgb)
    h, w = arr.shape[:2]
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
    
    grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.hypot(grad_x, grad_y)
    
    max_h = int(h * max_scan_pct)
    max_w = int(w * max_scan_pct)
    
    def is_artificial_band(slice_gray: np.ndarray, slice_mag: np.ndarray) -> bool:
        return float(np.std(slice_gray)) < 3.5 and float(np.mean(slice_mag)) < 3.5
        
    top = 0
    for r in range(max_h):
        if is_artificial_band(gray[r, :], mag[r, :]): top = r + 1
        else: break
        
    bottom = h
    for r in range(h - 1, h - max_h, -1):
        if is_artificial_band(gray[r, :], mag[r, :]): bottom = r
        else: break
        
    left = 0
    for c in range(max_w):
        if is_artificial_band(gray[:, c], mag[:, c]): left = c + 1
        else: break
        
    right = w
    for c in range(w - 1, w - max_w, -1):
        if is_artificial_band(gray[:, c], mag[:, c]): right = c
        else: break
        
    top = top if top >= 12 else 0
    bottom = bottom if (h - bottom) >= 12 else h
    left = left if left >= 12 else 0
    right = right if (w - right) >= 12 else w
    
    if top > 0 or bottom < h or left > 0 or right > w:
        return img_rgb.crop((left, top, right, bottom)), True
    return img_rgb, False

def prepare_image_pipeline(file_path: Path, max_dim: int = 1920):
    with Image.open(file_path) as raw_img:
        transposed = ImageOps.exif_transpose(raw_img)
        base = transposed if transposed is not None else raw_img
        rgb = base.convert('RGB')
        clean_img, was_cropped = detect_and_crop_artificial_bars_sobel(rgb)
        
        w, h = clean_img.size
        if max(w, h) > max_dim:
            scale = max_dim / max(w, h)
            final_img = clean_img.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
        else:
            final_img = clean_img.copy()
            
        del rgb, clean_img
        gc.collect()
        return final_img, was_cropped

print('✅ Pre-Procesamiento Sobel (Umbral 3.5) y Lanczos 1920 compilados.')

In [ ]:
# 4. Prompt Maestro Definitivo (pipeline_marinera_audit.md)
PROMPT_MAESTRO = """[TASK DIRECTIVE]
Professionally retouch this commercial fashion and jewelry photograph for the high-end e-commerce catalog of Marinera Power. Modify ONLY the background and ground surfaces.

[IMMUTABLE CORE: LIGHTING & COLOR CONSISTENCY]
- STUDIO BACKDROP: Place the entire subject on a seamless, clean, light warm-grey cyclorama studio sweep (uniform tone matching #E5E1DE, zero vignettes, zero harsh gradients).
- GROUND ANCHORING & CONTACT SHADOW: The vertical background and horizontal floor blend continuously. Anchor the subject to the ground plane with realistic, soft directional ambient occlusion shadows directly beneath shoe soles, bare feet, hemline, or display bases. Subject must feel solidly grounded, NEVER floating.
- COLOR FIDELITY & LIGHTING: Neutral daylight-balanced commercial studio strobes (5200K pure daylight simulation) with large octabox diffusion. High micro-contrast and exact textile saturation. Zero tungsten amber cast, zero yellow room bounce, zero cyan shifts.

[ANTI-PARQUET & FLOOR REPLACEMENT]
- Completely erase and replace all pre-existing flooring: eliminate wooden floorboards, parquet planks, dance studio tile joints, baseboards, and perspective floor lines. The #E5E1DE studio cyclorama sweep must completely replace the floor under and around the subject.

[CLIENT MANDATE: STRICT CROPPING PRESERVATION & NO OUTPAINTING]
- If a human subject or model is naturally cropped in the original frame (e.g., bust shot, medium portrait, torso, or legs ending at image borders), IT IS STRICTLY FORBIDDEN to alter the image framing, zoom out, change the canvas dimensions, or generate missing body limbs. Maintain the exact framing and crop of the original image untouched.

[NEGATIVE PROMPTS & ZERO-HALLUCINATION EXCLUSIONS]
- NEGATIVE: Do not alter, smooth, blur, or reconstruct original faces, eyes, fingers, hands, or bare feet.
- NEGATIVE: Do not deform, amputate, or merge the traditional white linen marinera handkerchief; keep its lace openwork, geometric drape, and tension crisp.
- NEGATIVE: For jewelry (filigree earrings, scapulars, brooches), do not blur or plasticize micro-twisted wirework; render as authentic 18k Peruvian yellow gold and 925 sterling silver with sharp specular highlights.
- NEGATIVE: If the subject is a garment on a hanger, dress form, or isolated jewelry, DO NOT hallucinate human limbs, heads, torsos, or artificial mannequins.
- NEGATIVE: Do not leave residual white borders, pillarbox bars, or uncropped Instagram frames."""

def edit_background_gemini(pil_img, prompt_text=PROMPT_MAESTRO, max_retries=3):
    buffered = io.BytesIO()
    pil_img.save(buffered, format='JPEG', quality=95)
    b64_image = base64.b64encode(buffered.getvalue()).decode('utf-8')

    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-image:generateContent?key={API_KEY}'
    payload = {
        'contents': [{
            'parts': [
                {'text': prompt_text},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_image}}
            ]
        }],
        'generationConfig': {
            'responseModalities': ['IMAGE'],
            'temperature': 0.1
        }
    }

    for intento in range(max_retries):
        try:
            res = requests.post(url, json=payload, timeout=90)
            if res.status_code == 200:
                parts = res.json()['candidates'][0]['content']['parts']
                for p in parts:
                    if 'inlineData' in p:
                        img_bytes = base64.b64decode(p['inlineData']['data'])
                        return Image.open(io.BytesIO(img_bytes)), None
            elif res.status_code in (429, 503):
                time.sleep(12 * (intento + 1))
                continue
            else:
                return None, f'HTTP {res.status_code}: {res.text[:120]}'
        except requests.exceptions.Timeout:
            if intento == max_retries - 1:
                return None, 'Timeout API (90s alcanzado)'
            time.sleep(5)
        except Exception as e:
            return None, str(e)
    return None, 'Fallo recurrente tras reintentos'

def audit_image_quality(orig_img, pro_img):
    buf1, buf2 = io.BytesIO(), io.BytesIO()
    orig_img.save(buf1, format='JPEG', quality=85)
    pro_img.save(buf2, format='JPEG', quality=85)
    b64_1 = base64.b64encode(buf1.getvalue()).decode('utf-8')
    b64_2 = base64.b64encode(buf2.getvalue()).decode('utf-8')

    eval_prompt = (
        'Actúa como Auditor Técnico de E-Commerce para Marinera Power. Evalúa rigurosamente:\n'
        '1. CERO OUTPAINTING: ¿Se conservó exactamente el encuadre y corte original sin inventar miembros faltantes?\n'
        '2. PARQUET/SUELO: ¿Se eliminó completamente el piso de madera/parquet y se integró el ciclorama #E5E1DE con oclusión?\n'
        '3. BORDES BLANCOS: ¿El fondo cubre el 100% sin barras de Instagram residuales?\n'
        '4. ANATOMÍA Y PAÑUELO: ¿Manos, dedos, pañuelo de marinera y filigrana están intactos?\n'
        'Responde estrictamente en JSON: {"score": 0-100, "aprobada": true/false, "motivo": "resumen"}'
    )
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={API_KEY}'
    payload = {
        'contents': [{
            'parts': [
                {'text': eval_prompt},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_1}},
                {'inlineData': {'mimeType': 'image/jpeg', 'data': b64_2}}
            ]
        }],
        'generationConfig': {'responseMimeType': 'application/json'}
    }
    try:
        res = requests.post(url, json=payload, timeout=25)
        if res.status_code == 200:
            return json.loads(res.json()['candidates'][0]['content']['parts'][0]['text'])
    except Exception:
        pass
    return {'score': 85, 'aprobada': True, 'motivo': 'Aprobada por omisión'}

print('✅ Prompt Maestro Definitivo y Motor QA cargados.')

In [ ]:
# 5. Módulo de Enrutamiento en Espejo & Checkpointing Persistente
def save_and_route_photo(pro_img, row_data, qa_result):
    row_id = str(row_data.get('ID', '0'))
    orig_name = str(row_data.get('Nombre_Archivo_Original', 'foto'))
    clean_name = orig_name.replace('.jpg', '').replace('.jpeg', '').replace('.png', '').replace('.heic', '').replace('.HEIC', '')
    score = qa_result.get('score', 85)
    aprobada = qa_result.get('aprobada', True) and (score >= 80)

    # Bóveda Central Permanente
    bov_path = BOVEDA_BACKUP_DIR / f'ID{row_id.zfill(3)}_{clean_name}_score{score}.jpg'
    pro_img.save(bov_path, quality=95)

    # Enrutamiento en Espejo
    ruta_origen = str(row_data.get('Ruta_Carpeta_Drive', row_data.get('Carpeta_Ruta_Original', '')))
    subrama = ruta_origen.replace('Fotos catalogo / ', '').replace('Fotos catalogo/', '').strip()

    if aprobada:
        target_dir = DIR_PRO_ROOT / subrama
        final_path = target_dir / f'{clean_name}_pro.jpg'
    else:
        target_dir = DIR_ERROR_ROOT / f'{subrama}_Error'
        final_path = target_dir / f'{clean_name}_QA_FALLIDA.jpg'

    target_dir.mkdir(parents=True, exist_ok=True)
    pro_img.save(final_path, quality=95)
    return str(final_path), str(bov_path), aprobada

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        try:
            with open(CHECKPOINT_FILE, 'r') as f:
                return json.load(f)
        except Exception:
            pass
    return {'processed_ids': []}

def save_checkpoint(processed_ids):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({'processed_ids': processed_ids, 'last_update': time.strftime('%Y-%m-%d %H:%M:%S')}, f, indent=2)

print('✅ Enrutamiento en Espejo y Checkpointing persistente listos.')

In [ ]:
# 6. CANARY TEST DETERMINISTA (5 Casos Críticos de Fuego)
canary_targets = [
    {
        'id_test': 'CANARY_1_BARRAS_BLANCAS',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Lila con cinta morada',
        'orig_name': 'foto (3).png',
        'caso': 'Recorte Sobel de barras laterales blancas de Instagram (193px)'
    },
    {
        'id_test': 'CANARY_2_PARQUET_SALON',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Lila con cinta morada',
        'orig_name': '34a63e97 PRO.jpg',
        'caso': 'Reemplazo total de suelo de parquet por ciclorama #E5E1DE con oclusión'
    },
    {
        'id_test': 'CANARY_3_PAÑUELO_MANO',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Coral con cinta blanca',
        'orig_name': 'd72049af PRO.jpg',
        'caso': 'Blindaje de mano y pañuelo de marinera al viento (cero amputación)'
    },
    {
        'id_test': 'CANARY_4_FILIGRANA_JOYA',
        'subpath': 'Para ella/Accesorios/Aretes para Marinera',
        'orig_name': '1.jpg',
        'caso': 'Macro joyería: Oro peruano 18k y filigrana calada con brillo especular'
    },
    {
        'id_test': 'CANARY_5_CORTE_ENQUADRE',
        'subpath': 'Para ella/Falda/Falda de Ensayo/Faldas de Ensayo de razo 1 bobo/Falda para Ensayo de Razo - Morada con cinta blanca',
        'orig_name': '1841c6d0 PRO.jpg',
        'caso': 'Mandato Cliente: Cero outpainting en plano recortado (conservar encuadre)'
    }
]

print(f"🦅 INICIANDO CANARY TEST DETERMINISTA (Ruta Base: {DIR_ORIGEN_ROOT})...\n")
canary_results = []

for idx, tgt in enumerate(canary_targets, start=1):
    test_file = resolve_photo_path(tgt['subpath'], tgt['orig_name'])
    
    if not test_file or not test_file.exists():
        print(f"⚠️ [{idx}/5] No se localizó archivo para {tgt['id_test']}: {tgt['orig_name']}")
        continue

    print(f"▶️ [{idx}/5] Evaluando {tgt['id_test']}: {test_file.name}...")
    print(f"   🎯 Caso a probar: {tgt['caso']}")
    
    t0 = time.time()
    try:
        prep_img, was_cropped = prepare_image_pipeline(test_file)
        pro_img, err = edit_background_gemini(prep_img)
        
        if pro_img:
            qa = audit_image_quality(prep_img, pro_img)
            final_p, bov_p, aprobada = save_and_route_photo(pro_img, {
                'ID': f'CANARY_{idx}',
                'Nombre_Archivo_Original': test_file.name,
                'Ruta_Carpeta_Drive': tgt['subpath']
            }, qa)
            t_tot = time.time() - t0
            
            print(f"   ✅ Score QA: {qa.get('score')}/100 | Aprobada: {aprobada} | Total: {t_tot:.1f}s (Sobel Crop: {was_cropped})")
            print(f"   📝 Motivo QA: {qa.get('motivo')}")
            print(f"   📁 Guardada en: {final_p}")
            canary_results.append({'test': tgt['id_test'], 'score': qa.get('score'), 'aprobada': aprobada, 'file': test_file.name})
            display(pro_img)
        else:
            print(f"   ❌ Fallo API: {err}")
    except Exception as exc:
        print(f"   ❌ Error procesando {test_file.name}: {exc}")
    time.sleep(3)

aprobadas = sum(1 for c in canary_results if c.get('aprobada'))
print(f"\n🏆 RESUMEN CANARY TEST: {aprobadas}/{len(canary_results)} Aprobadas.")
if aprobadas >= 4:
    print("🚀 ¡CANARY TEST SUPERADO CON ÉXITO! Las fotos cumplen todos los Quality Gates del Red Team.")



In [ ]:
# ==============================================================================
# 🎯 PASO 7: PROCESAMIENTO EN LOTES DE 20 FOTOS (HUMAN-IN-THE-LOOP)
# ==============================================================================
# 🔒 SEGURO DE ACTIVACIÓN MANUAL:
# Para ejecutar el Lote 1 (20 fotos), cambia esta variable a True:
EJECUTAR_LOTE_PRODUCCION = False

CHUNK_SIZE = 20  # Lote estricto de 20 fotos solicitado por Carlos Carbajal

if not EJECUTAR_LOTE_PRODUCCION:
    print("🔒 [PASO 7 EN ESPERA] Seguro de activación manual activo.")
    print("👉 Este paso NO se ejecutará al pulsar 'Ejecutar todo'.")
    print(f"👉 Para procesar el siguiente lote de {CHUNK_SIZE} fotos:")
    print("   1. Cambia arriba: EJECUTAR_LOTE_PRODUCCION = True")
    print("   2. Presiona únicamente el botón Play de esta celda.")
else:
    print(f"🚀 INICIANDO LOTE DE {CHUNK_SIZE} FOTOS (Mesa de Control Oficial)...")
    
    # 1. Cargar datos de la pestaña oficial: Control_Total_Drive_V2
    sheet_data = sheet_control.get_all_records()
    headers = sheet_control.row_values(1)
    
    # Column indices (1-based for gspread)
    col_estado_idx = headers.index('Estado') + 1 if 'Estado' in headers else 8
    
    # Asegurar columnas para feedback de Carlos
    if 'Feedback_Carlos' not in headers:
        sheet_control.update_cell(1, len(headers) + 1, 'Feedback_Carlos')
        sheet_control.update_cell(1, len(headers) + 2, 'Detalle_Error_Carlos')
        headers = sheet_control.row_values(1)
        
    col_feedback_idx = headers.index('Feedback_Carlos') + 1
    col_detalle_idx = headers.index('Detalle_Error_Carlos') + 1
    
    # 2. Cargar Checkpoint
    checkpoint = load_checkpoint()
    processed_set = set(checkpoint.get('processed_ids', []))
    print(f"📦 Fotos ya aseguradas en checkpoint previo: {len(processed_set)}")
    
    # 3. Filtrar pendientes
    pendientes = []
    for idx, row in enumerate(sheet_data, start=2):
        row_id = str(row.get('ID', ''))
        estado = str(row.get('Estado', '')).upper()
        if row_id and (row_id not in processed_set) and ('PRO_VERIFICADO' not in estado):
            pendientes.append((idx, row))
            
    print(f"📊 Total fotos pendientes en catálogo: {len(pendientes)}")
    chunk_to_process = pendientes[:CHUNK_SIZE]
    print(f"🎯 Procesando lote de {len(chunk_to_process)} fotos ahora...\n")
    
    exitosas_chunk = 0
    rechazadas_chunk = 0
    
    for i, (row_idx, row_data) in enumerate(chunk_to_process, start=1):
        row_id = str(row_data.get('ID', '0'))
        orig_name = str(row_data.get('Nombre_Archivo_Original', ''))
        ruta_origen = str(row_data.get('Ruta_Carpeta_Drive', row_data.get('Carpeta_Ruta_Original', '')))
        
        print(f"[{i}/{len(chunk_to_process)}] Foto ID {row_id}: {orig_name}")
        
        # Localización ultra-robusta
        foto_path = resolve_photo_path(ruta_origen, orig_name)
        if not foto_path or not foto_path.exists():
            print(f"   ⚠️ Archivo físico no hallado en Drive. Saltando.")
            continue
            
        t0 = time.time()
        try:
            # Pre-procesamiento Sobel
            prep_img, was_cropped = prepare_image_pipeline(foto_path)
            
            # Generación IA
            pro_img, err = edit_background_gemini(prep_img)
            
            if pro_img:
                # Auditoría de Calidad
                qa = audit_image_quality(prep_img, pro_img)
                final_path, bov_path, aprobada = save_and_route_photo(pro_img, row_data, qa)
                
                # Registro en Sheet
                nuevo_estado = 'PRO_PENDIENTE_FEEDBACK_CARLOS' if aprobada else 'FALLIDA_QA'
                sheet_control.update_cell(row_idx, col_estado_idx, nuevo_estado)
                
                if aprobada:
                    exitosas_chunk += 1
                    print(f"   ✅ Aprobada QA (Score {qa.get('score')}/100) -> Guardada en PRO y Bóveda")
                else:
                    rechazadas_chunk += 1
                    print(f"   ⚠️ Rechazada QA (Score {qa.get('score')}/100) -> Guardada en Carpeta Erróneas")
                    
                display(pro_img)
            else:
                sheet_control.update_cell(row_idx, col_estado_idx, 'ERROR_GENERACION_API')
                print(f"   ❌ Fallo API Gemini: {err}")
                
        except Exception as exc:
            print(f"   ❌ Excepción aislada en ID {row_id}: {exc}")
        finally:
            gc.collect()
            
        processed_set.add(row_id)
        save_checkpoint(list(processed_set))
        time.sleep(3)
        
    print(f"\n==============================================================================")
    print(f"🎉 LOTE DE {len(chunk_to_process)} FOTOS COMPLETADO!")
    print(f"📊 Resumen: {exitosas_chunk} aprobadas por QA | {rechazadas_chunk} en carpeta erróneas")
    print(f"📁 Todas las fotos tienen copia de seguridad en: _BOVEDA_HISTORIAL_BACKUP")
    print(f"🌐 Tablero de Control Web: https://carloscarbajallinan.github.io/marinera-studio-pro/")
    print(f"📋 Hoja Oficial en Sheets: {sheet_control.title}")
    print(f"👉 Asigna tu feedback (APROBADO_CARLOS / CORREGIR_CARLOS) en el Tablero o Sheet")
    print(f"   antes de correr el siguiente lote.")
    print(f"==============================================================================")

